In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install pathlib
%pip install scikit-learn
%pip install imbalanced-learn
%pip install scipy
%pip install xgboost

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [1]:
import os
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"La máquina tiene {num_cores} núcleos disponibles.")
hilos_optimos = str(min(num_cores, 64)) 
print(f"Configurando OpenBLAS para usar {hilos_optimos} hilos máximos...")
os.environ['OPENBLAS_NUM_THREADS'] = hilos_optimos
os.environ['MKL_NUM_THREADS'] = hilos_optimos
os.environ['OMP_NUM_THREADS'] = hilos_optimos

La máquina tiene 224 núcleos disponibles.
Configurando OpenBLAS para usar 64 hilos máximos...


In [2]:
import numpy as np
import pandas as pd
import itertools
import json
from pathlib import Path
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.svm import SVC
from sklearn.metrics import (f1_score, accuracy_score, recall_score,classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.combine import SMOTEENN


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import xgboost as xgb
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import gc

In [3]:
X = pd.read_pickle("Sets_Xy/X.pkl")
y = pd.read_pickle("Sets_Xy/y.pkl")

In [4]:
from sklearn.model_selection import train_test_split

#Division estratificada para muestras de cada clase a nivel de cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, 
    random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, 
    random_state=42)

In [5]:
#Encoding de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)
y_test = le.transform(y_test)

mapeo_labels = pd.DataFrame({
    "label_original": le.classes_,
    "label_encoded": range(len(le.classes_))
})
print("Mapeo de etiquetas:\n", mapeo_labels)
class_names = le.classes_

Mapeo de etiquetas:
                label_original  label_encoded
0                      BENIGN              0
1                         Bot              1
2                        DDoS              2
3               DoS GoldenEye              3
4                    DoS Hulk              4
5            DoS Slowhttptest              5
6               DoS slowloris              6
7                 FTP-Patator              7
8                  Heartbleed              8
9                Infiltration              9
10                   PortScan             10
11                SSH-Patator             11
12    Web Attack  Brute Force             12
13  Web Attack  Sql Injection             13
14            Web Attack  XSS             14


In [4]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, method='tree', k=50, class_weight='balanced', random_state=42):
        self.method = method
        self.k = k
        self.class_weight = class_weight
        self.random_state = random_state
        self.selector = None
        self.feature_names_ = None
        self.support_mask_ = None

    def fit(self, X, y):
        self.feature_names_ = X.columns.tolist() if hasattr(X, 'columns') else list(range(X.shape[1]))
        
        X_array = X.values if hasattr(X, 'values') else X
        y_array = y.values if hasattr(y, 'values') else y
        
        total_features = X_array.shape[1]
        actual_k = min(self.k, total_features)
        
        if self.method == 'none':
            self.support_mask_ = np.ones(total_features, dtype=bool)
            return self

        if self.method == 'f_classif':
            self.selector = SelectKBest(score_func=f_classif, k=actual_k)
            self.selector.fit(X_array, y_array)
            self.support_mask_ = self.selector.get_support()

        elif self.method == 'tree':
            estimator = ExtraTreesClassifier(
                n_estimators=100,               
                max_depth=20,                   
                class_weight=self.class_weight,
                criterion='gini',               
                random_state=self.random_state, 
                n_jobs=-1
            )
            estimator.fit(X_array, y_array)
            importances = estimator.feature_importances_
            
            top_k_indices = np.argsort(importances)[-actual_k:]
            
            self.support_mask_ = np.zeros(total_features, dtype=bool)
            self.support_mask_[top_k_indices] = True

        return self

    def transform(self, X):
        if self.method == 'none':
            return X
            
        X_array = X.values if hasattr(X, 'values') else X
        X_transformed = X_array[:, self.support_mask_]

        if hasattr(X, 'columns'):
            selected_features = np.array(self.feature_names_)[self.support_mask_].tolist()
            return pd.DataFrame(X_transformed, columns=selected_features, index=X.index)
        else:
            return X_transformed

In [7]:
#Carga de datos post oversampling
import joblib

X_train_none = joblib.load("Sets_Oversampling/X_train_none.joblib")
y_train_none = joblib.load("Sets_Oversampling/y_train_none.joblib")

X_train_smote = joblib.load("Sets_Oversampling/X_train_smote.joblib")
y_train_smote = joblib.load("Sets_Oversampling/y_train_smote.joblib")

X_train_smote_enn = joblib.load("Sets_Oversampling/X_train_smote_enn.joblib")
y_train_smote_enn = joblib.load("Sets_Oversampling/y_train_smote_enn.joblib")

X_train_smote_tomek = joblib.load("Sets_Oversampling/X_train_smote_tomek.joblib")
y_train_smote_tomek = joblib.load("Sets_Oversampling/y_train_smote_tomek.joblib")

X_val = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian.pkl")
y_val = pd.DataFrame(y_val)

X_test = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian.pkl")
y_test = pd.DataFrame(y_test)

In [8]:
X_train_none_2 = joblib.load("Sets_Oversampling_2/X_train_none.joblib")
y_train_none_2 = joblib.load("Sets_Oversampling_2/y_train_none.joblib")

X_train_smote_2 = joblib.load("Sets_Oversampling_2/X_train_smote.joblib")
y_train_smote_2 = joblib.load("Sets_Oversampling_2/y_train_smote.joblib")

X_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/X_train_smote_enn.joblib")
y_train_smote_enn_2 = joblib.load("Sets_Oversampling_2/y_train_smote_enn.joblib")

X_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/X_train_smote_tomek.joblib")
y_train_smote_tomek_2 = joblib.load("Sets_Oversampling_2/y_train_smote_tomek.joblib")


In [5]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight

def balanced_class_weight(y):
    classes = np.unique(y)
    weights = compute_class_weight('balanced', classes=classes, y=y)
    return dict(zip(classes, weights))

def polynomial_class_weight(y, alpha=0.25):
    classes, counts = np.unique(y, return_counts=True)
    weights = 1.0 / np.power(counts, alpha)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def log_ratio_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    total = np.sum(counts)
    weights = np.log(total / counts)
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def effective_samples_class_weight(y, beta=0.999):
    classes, counts = np.unique(y, return_counts=True)
    effective_num = 1.0 - np.power(beta, counts)
    weights = (1.0 - beta) / effective_num
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def cost_sensitive_class_weight(y):
    classes, counts = np.unique(y, return_counts=True)
    max_count = np.max(counts)
    weights = max_count / counts
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

def focal_class_weight_improved(y, gamma=2.0):

    classes, counts = np.unique(y, return_counts=True)
    probs = counts / np.sum(counts)
    weights = (1 - probs) ** gamma / probs
    weights = weights / np.mean(weights)
    return dict(zip(classes, weights))

In [ ]:
train_datasets = {
    'none': (X_train_none, y_train_none.values.ravel() if isinstance(y_train_none, pd.DataFrame) else y_train_none.ravel()),
    'smote_enn': (X_train_smote_enn, y_train_smote_enn.values.ravel() if isinstance(y_train_smote_enn, pd.DataFrame) else y_train_smote_enn.ravel()),
    'smote_tomek': (X_train_smote_tomek, y_train_smote_tomek.values.ravel() if isinstance(y_train_smote_tomek, pd.DataFrame) else y_train_smote_tomek.ravel())
}

y_val_1d = y_val.values.ravel() if isinstance(y_val, pd.DataFrame) else y_val.ravel()
y_test_1d = y_test.values.ravel() if isinstance(y_test, pd.DataFrame) else y_test.ravel()

In [11]:
train_datasets_2 = {
    'none': (X_train_none_2, y_train_none_2.values.ravel() if isinstance(y_train_none_2, pd.DataFrame) else y_train_none_2.ravel()),
    'smote': (X_train_smote_2, y_train_smote_2.values.ravel() if isinstance(y_train_smote_2, pd.DataFrame) else y_train_smote_2.ravel()),
    'smote_enn': (X_train_smote_enn_2, y_train_smote_enn_2.values.ravel() if isinstance(y_train_smote_enn_2, pd.DataFrame) else y_train_smote_enn_2.ravel()),
    'smote_tomek': (X_train_smote_tomek_2, y_train_smote_tomek_2.values.ravel() if isinstance(y_train_smote_tomek_2, pd.DataFrame) else y_train_smote_tomek_2.ravel())
}

In [7]:
import joblib
X_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_none.joblib")
y_train_none_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_none.joblib")

X_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote.joblib")
y_train_smote_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote.joblib")

X_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_enn.joblib")
y_train_smote_enn_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_enn.joblib")

X_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/X_train_smote_tomek.joblib")
y_train_smote_tomek_grouped = joblib.load("Sets_Oversampling_Grouped/y_train_smote_tomek.joblib")

X_val_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_val_scaled_bayesian_grouped.pkl")
X_test_imp_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/X_test_scaled_bayesian_grouped.pkl")

y_val_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_val_grouped.pkl")
y_test_grouped = pd.read_pickle("Sets_Post_Scaled_Imp/y_test_grouped.pkl")

In [8]:
train_datasets_grouped = {
    'none': (X_train_none_grouped, y_train_none_grouped.values.ravel() if isinstance(y_train_none_grouped, pd.DataFrame) else y_train_none_grouped.ravel()),
    'smote': (X_train_smote_grouped, y_train_smote_grouped.values.ravel() if isinstance(y_train_smote_grouped, pd.DataFrame) else y_train_smote_grouped.ravel()),
    'smote_enn': (X_train_smote_enn_grouped, y_train_smote_enn_grouped.values.ravel() if isinstance(y_train_smote_enn_grouped, pd.DataFrame) else y_train_smote_enn_grouped.ravel()),
    'smote_tomek': (X_train_smote_tomek_grouped, y_train_smote_tomek_grouped.values.ravel() if isinstance(y_train_smote_tomek_grouped, pd.DataFrame) else y_train_smote_tomek_grouped.ravel())
}

In [9]:
def save_confusion_matrix_xgb(log_folder, y_true, y_pred, trial_number, dataset_name, phase="val"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'confusion matrix - {dataset_name} trial {trial_number} ({phase})')
    plt.ylabel('true label')
    plt.xlabel('predicted label')
    plt.savefig(f'{log_folder}/{dataset_name}_trial_{trial_number}_xgb_{phase}_cm.png', bbox_inches='tight')
    plt.close()

In [ ]:
import os
import optuna
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

log_folder = "Logs_XGBoost_Baseline_1"
os.makedirs(log_folder, exist_ok=True)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

datasets_to_test = ['none', 'smote', 'smote_enn', 'smote_tomek']
resultados_globales = {}

for current_dataset in datasets_to_test:
    print(f"\niniciando estudio enfocado en set agrupado: {current_dataset}")
    
    def objective_xgb(trial):
        x_train_raw, y_train_raw = train_datasets_grouped[current_dataset]
        total_features = x_train_raw.shape[1]

        weight_func_name = trial.suggest_categorical('weight_func', ['none', 'balanced', 'log_ratio', 'cost_sensitive'])
        
        if weight_func_name == 'none':
            weight_dict_fs = None
            sample_weights_xgb = None
            
        elif weight_func_name == 'balanced':
            weight_dict_fs = 'balanced'
            sample_weights_xgb = compute_sample_weight('balanced', y=y_train_raw)
            
        else:
            if weight_func_name == 'log_ratio':
                weight_dict_fs = log_ratio_class_weight(y_train_raw)
            elif weight_func_name == 'cost_sensitive':
                weight_dict_fs = cost_sensitive_class_weight(y_train_raw)

            web_boost = trial.suggest_float('web_boost', 1.0, 3.0) 
            bot_boost = trial.suggest_float('bot_boost', 1.5, 3.5) 
            
            if 12 in weight_dict_fs: weight_dict_fs[12] *= web_boost
            if 1 in weight_dict_fs: weight_dict_fs[1] *= bot_boost
                
            map_func = np.vectorize(lambda x: weight_dict_fs.get(x, 1.0))
            sample_weights_xgb = map_func(y_train_raw)

        fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
        
        if fs_method == 'none':
            k_features = total_features
        else:
            k_features = trial.suggest_int('k_features', 20, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features, class_weight=weight_dict_fs) 
        x_train_fs = selector.fit_transform(x_train_raw, y_train_raw)
        x_val_fs = selector.transform(X_val_imp_grouped)
        
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 150, 400),
            'max_depth': trial.suggest_int('max_depth', 6, 15), 
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            'objective': 'multi:softmax',
            'num_class': 13,
            'random_state': 42
        }
        
        model = xgb.XGBClassifier(**xgb_params)
        
        model.fit(x_train_fs, y_train_raw, sample_weight=sample_weights_xgb)
        
        y_pred = model.predict(x_val_fs)
        
        f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
        report = classification_report(y_val_grouped, y_pred, zero_division=0)
        
        save_confusion_matrix_xgb(log_folder,y_val_grouped, y_pred, trial.number, current_dataset, phase="val")
        
        log_msg = f"""dataset: {current_dataset} | trial: {trial.number}
fs method: {fs_method} | k features: {k_features}
params xgb: {trial.params}

f1 macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)
            
        return f1_mac

    study_name = f"xgb_opt_grouped_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_xgb, n_trials=100, n_jobs=1)
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params
    }
    
    best_log_msg = f"""mejor trial para dataset agrupado (xgboost): {current_dataset}
trial numero: {study.best_trial.number}
f1 macro alcanzado: {study.best_value:.4f}
hiperparametros:
{study.best_params}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)

    print(f"\nresultados finales para {current_dataset}")
    print(f"mejor trial: {study.best_trial.number}")
    print(f"mejor f1 macro: {study.best_value:.4f}")
    print(f"mejores params: {study.best_params}\n")

informe_final = "informe final: competencia de datasets agrupados (xgboost)\n"

for dataset, metricas in resultados_globales.items():
    informe_final += f"dataset: {dataset}\n"
    informe_final += f"- mejor trial: {metricas['best_trial']}\n"
    informe_final += f"- f1 macro alcanzado: {metricas['best_f1_macro']:.4f}\n"
    informe_final += f"- hiperparametros ganadores:\n"
    for param, value in metricas['best_params'].items():
        informe_final += f"* {param}: {value}\n"

ganador_absoluto = max(resultados_globales, key=lambda x: resultados_globales[x]['best_f1_macro'])
informe_final += f"\nganador absoluto: {ganador_absoluto} (f1 macro: {resultados_globales[ganador_absoluto]['best_f1_macro']:.4f})\n"

print(informe_final)

with open(f"{log_folder}/resumen_global_resultados.txt", 'w') as f:
    f.write(informe_final)

print(f"el informe final ha sido guardado en {log_folder}/resumen_global_resultados.txt")

[I 2026-03-09 16:41:18,646] A new study created in memory with name: xgb_opt_grouped_none



iniciando estudio enfocado en set agrupado: none


[I 2026-03-09 16:41:37,193] Trial 0 finished with value: 0.9378755991469474 and parameters: {'weight_func': 'log_ratio', 'web_boost': 2.2048068932838767, 'bot_boost': 1.9922156872911738, 'fs_method': 'tree', 'k_features': 51, 'n_estimators': 175, 'max_depth': 14, 'learning_rate': 0.019181078383558047, 'subsample': 0.9939469660964133, 'colsample_bytree': 0.8254151137360504}. Best is trial 0 with value: 0.9378755991469474.
[I 2026-03-09 16:42:00,533] Trial 1 finished with value: 0.9188458470924818 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 54, 'n_estimators': 261, 'max_depth': 10, 'learning_rate': 0.015058824819604178, 'subsample': 0.7969410182403996, 'colsample_bytree': 0.9124281379258941}. Best is trial 0 with value: 0.9378755991469474.
[I 2026-03-09 16:42:14,471] Trial 2 finished with value: 0.7244570952150051 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 2.866350479583524, 'bot_boost': 2.991438473290504, 'fs_method': 'tree', 'k_fea


resultados finales para none
mejor trial: 65
mejor f1 macro: 0.9617
mejores params: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 59, 'n_estimators': 387, 'max_depth': 9, 'learning_rate': 0.1505830237684683, 'subsample': 0.9988985881667181, 'colsample_bytree': 0.6463159010113986}


iniciando estudio enfocado en set agrupado: smote


[I 2026-03-09 17:21:03,143] Trial 0 finished with value: 0.9521438562098451 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 378, 'max_depth': 12, 'learning_rate': 0.03680873188474869, 'subsample': 0.6068822422914893, 'colsample_bytree': 0.7015304781124833}. Best is trial 0 with value: 0.9521438562098451.
[I 2026-03-09 17:21:28,979] Trial 1 finished with value: 0.9236192168209821 and parameters: {'weight_func': 'log_ratio', 'web_boost': 2.217782206229149, 'bot_boost': 2.7494684625220938, 'fs_method': 'tree', 'k_features': 49, 'n_estimators': 251, 'max_depth': 10, 'learning_rate': 0.02633660773116107, 'subsample': 0.7596925216186603, 'colsample_bytree': 0.7596728228490404}. Best is trial 0 with value: 0.9521438562098451.
[I 2026-03-09 17:21:50,629] Trial 2 finished with value: 0.9140050207001362 and parameters: {'weight_func': 'balanced', 'fs_method': 'none', 'n_estimators': 246, 'max_depth': 14, 'learning_rate': 0.031207335493366985, 'subsample': 0.716765236


resultados finales para smote
mejor trial: 65
mejor f1 macro: 0.9529
mejores params: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 377, 'max_depth': 7, 'learning_rate': 0.0922100112806016, 'subsample': 0.6213367240678677, 'colsample_bytree': 0.859904818858891}


iniciando estudio enfocado en set agrupado: smote_enn


[I 2026-03-09 17:59:52,592] Trial 0 finished with value: 0.9257425856842686 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 57, 'n_estimators': 187, 'max_depth': 13, 'learning_rate': 0.011396412280312155, 'subsample': 0.9990272608593455, 'colsample_bytree': 0.6226153419240232}. Best is trial 0 with value: 0.9257425856842686.
[I 2026-03-09 18:00:30,694] Trial 1 finished with value: 0.948333934152289 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 66, 'n_estimators': 393, 'max_depth': 15, 'learning_rate': 0.06751687718843075, 'subsample': 0.7822692727939909, 'colsample_bytree': 0.8373667752129365}. Best is trial 1 with value: 0.948333934152289.
[I 2026-03-09 18:00:52,467] Trial 2 finished with value: 0.9166630333960085 and parameters: {'weight_func': 'log_ratio', 'web_boost': 2.268544433903277, 'bot_boost': 2.0185951315698194, 'fs_method': 'tree', 'k_features': 53, 'n_estimators': 230, 'max_depth': 8, 'learning_rate': 0.021150


resultados finales para smote_enn
mejor trial: 70
mejor f1 macro: 0.9509
mejores params: {'weight_func': 'balanced', 'fs_method': 'none', 'n_estimators': 353, 'max_depth': 15, 'learning_rate': 0.18643013865156902, 'subsample': 0.8758674229116665, 'colsample_bytree': 0.9826219307845969}


iniciando estudio enfocado en set agrupado: smote_tomek


[I 2026-03-09 18:40:40,796] Trial 0 finished with value: 0.8472441847818721 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 1.0978147519927222, 'bot_boost': 2.3416958876022447, 'fs_method': 'none', 'n_estimators': 302, 'max_depth': 11, 'learning_rate': 0.1459791292052643, 'subsample': 0.9285148422289317, 'colsample_bytree': 0.738603454733671}. Best is trial 0 with value: 0.8472441847818721.
[I 2026-03-09 18:41:10,559] Trial 1 finished with value: 0.9435539329801561 and parameters: {'weight_func': 'none', 'fs_method': 'tree', 'k_features': 57, 'n_estimators': 264, 'max_depth': 13, 'learning_rate': 0.012901591243160574, 'subsample': 0.8014428324392674, 'colsample_bytree': 0.6800790161751695}. Best is trial 1 with value: 0.9435539329801561.
[I 2026-03-09 18:41:32,824] Trial 2 finished with value: 0.7923553104622066 and parameters: {'weight_func': 'cost_sensitive', 'web_boost': 1.41864327391792, 'bot_boost': 2.103759223032418, 'fs_method': 'tree', 'k_features': 36, 'n_estima


resultados finales para smote_tomek
mejor trial: 38
mejor f1 macro: 0.9544
mejores params: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 161, 'max_depth': 8, 'learning_rate': 0.16975347514334424, 'subsample': 0.9337508865655523, 'colsample_bytree': 0.7759752037198928}

informe final: competencia de datasets agrupados (xgboost)
dataset: none
- mejor trial: 65
- f1 macro alcanzado: 0.9617
- hiperparametros ganadores:
* weight_func: none
* fs_method: tree
* k_features: 59
* n_estimators: 387
* max_depth: 9
* learning_rate: 0.1505830237684683
* subsample: 0.9988985881667181
* colsample_bytree: 0.6463159010113986
dataset: smote
- mejor trial: 65
- f1 macro alcanzado: 0.9529
- hiperparametros ganadores:
* weight_func: none
* fs_method: none
* n_estimators: 377
* max_depth: 7
* learning_rate: 0.0922100112806016
* subsample: 0.6213367240678677
* colsample_bytree: 0.859904818858891
dataset: smote_enn
- mejor trial: 70
- f1 macro alcanzado: 0.9509
- hiperparametros ganadores:
* w

In [ ]:
print("Iniciando evaluación final en Test Set para los mejores modelos de XGBoost...\n")

log_folder = "Logs_XGBoost_Baseline_1"

best_xgb_configs = {
    'none': {
        'trial': 65, 
        'fs_method': 'tree', 
        'k_features': 59, 
        'weight_func': 'none',
        'xgb_params': {
            'n_estimators': 387, 'max_depth': 9, 'learning_rate': 0.1505830237684683, 
            'subsample': 0.9988985881667181, 'colsample_bytree': 0.6463159010113986, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote': {
        'trial': 65, 
        'fs_method': 'none', 
        'k_features': 80,
        'weight_func': 'none',
        'xgb_params': {
            'n_estimators': 377, 'max_depth': 7, 'learning_rate': 0.0922100112806016, 
            'subsample': 0.6213367240678677, 'colsample_bytree': 0.859904818858891, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote_enn': {
        'trial': 70, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'balanced',
        'xgb_params': {
            'n_estimators': 353, 'max_depth': 15, 'learning_rate': 0.18643013865156902, 
            'subsample': 0.8758674229116665, 'colsample_bytree': 0.9826219307845969, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote_tomek': {
        'trial': 38, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'none',
        'xgb_params': {
            'n_estimators': 161, 'max_depth': 8, 'learning_rate': 0.16975347514334424, 
            'subsample': 0.9337508865655523, 'colsample_bytree': 0.7759752037198928, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softmax', 
            'num_class': 13, 'random_state': 42
        }
    }
}

for current_dataset, config in best_xgb_configs.items():
    print(f"Evaluando modelo XGBoost campeón para: {current_dataset.upper()}")
    
    x_train_best, y_train_best = train_datasets_grouped[current_dataset]
    
    weight_dict_fs = None
    sample_weights_xgb = None
    
    if config['weight_func'] == 'balanced':
        weight_dict_fs = 'balanced'
        sample_weights_xgb = compute_sample_weight('balanced', y=y_train_best)
        print("Aplicando pesos 'balanced'...")

    print(f"Aplicando Feature Selection: {config['fs_method']} (k={config['k_features']})...")
    final_selector = FeatureSelector(method=config['fs_method'], k=config['k_features'], class_weight=weight_dict_fs)
    x_train_final = final_selector.fit_transform(x_train_best, y_train_best)
    x_test_final = final_selector.transform(X_test_imp_grouped) 
    
    x_train_final = np.ascontiguousarray(x_train_final, dtype=np.float32)
    y_train_best = np.ascontiguousarray(y_train_best, dtype=np.int32)
    x_test_final = np.ascontiguousarray(x_test_final, dtype=np.float32)
    if sample_weights_xgb is not None:
        sample_weights_xgb = np.ascontiguousarray(sample_weights_xgb, dtype=np.float32)

    print("Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..")
    final_xgb = xgb.XGBClassifier(**config['xgb_params'])
    
    if sample_weights_xgb is not None:
        final_xgb.fit(x_train_final, y_train_best, sample_weight=sample_weights_xgb)
    else:
        final_xgb.fit(x_train_final, y_train_best)
    
    print("Evaluando en el set de test puro...")
    y_pred_test = final_xgb.predict(x_test_final)
    
    test_f1_mac = f1_score(y_test_grouped, y_pred_test, average='macro')
    test_report = classification_report(y_test_grouped, y_pred_test, zero_division=0)
    
    print(f"Resultados finales - F1 macro: {test_f1_mac:.4f}\n")
    
    save_confusion_matrix_xgb(log_folder,
        y_true=y_test_grouped, 
        y_pred=y_pred_test, 
        trial_number=config['trial'], 
        dataset_name=current_dataset, 
        phase="test_final"
    )
    
    log_msg = f"""Evaluación final con Test
Modelo: XGBoost
Dataset: {current_dataset} | Feature Selection: {config['fs_method']} (k={config['k_features']})
Weight Function: {config['weight_func']}
Params XGB: {config['xgb_params']}

F1 macro (Test): {test_f1_mac:.4f}

Métricas por clase (Test):
{test_report}
"""
    log_filename = f"{log_folder}/xgb_final_test_metrics_{current_dataset}.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
        
    del final_xgb
    del x_train_final
    del x_test_final
    gc.collect()

print("Pipeline de pruebas completado con éxito para XGBoost.")

Iniciando evaluación final en Test Set para los campeones de XGBoost...

Evaluando modelo XGBoost campeón para: NONE
Aplicando Feature Selection: tree (k=59)...


Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9038

Evaluando modelo XGBoost campeón para: SMOTE
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9204

Evaluando modelo XGBoost campeón para: SMOTE_ENN
Aplicando pesos 'balanced'...
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9123

Evaluando modelo XGBoost campeón para: SMOTE_TOMEK
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando en el set de test puro...
Resultados finales - F1 macro: 0.9212

Pipeline de pruebas completado con éxito para XGBoost.
Revisa la carpeta Logs_XGBoost_Baseline_2/ para ver las matrices y los reportes.


In [16]:
#Revisando los resultados hasta el momento he identificado que el problema para mejorar aun mas la clasificacion se encuentra en la clase 1 (Ataque Bot) y las clases 8 y 9 (Heartbleed y Infiltration)

#Para esto voy a hacer tests cambiando el enfoque de XGBoost al emplear multi:softprob en vez de multi:softmax para obtener probabilidades de clase con base en las cuales se pueda configurar el modelo. Caso clase 1

#Tambien siguiendo la recomendacion de mi tutor y de lo definido en mi propuesta voy a aplicar Focal Loss para penalizar fallos en clases minoritarias y mejorar la especializacion sobre las mismas. Caso clases 8 y 9

#Ademas el bot boost que emplee para la clase 1 voy a restringirlo con base en mis resultados acotando su intervalo para Optuna

In [ ]:
import os
import optuna
import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_sample_weight
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

log_folder = "Logs_XGBoost_Baseline_2"
os.makedirs(log_folder, exist_ok=True)

datasets_to_test = ['none', 'smote', 'smote_enn', 'smote_tomek']
resultados_globales = {}

for current_dataset in datasets_to_test:
    print(f"\nIniciando estudio enfocado en set agrupado con XGBoost: {current_dataset}")
    
    def objective_xgb(trial):
        x_train_raw, y_train_raw = train_datasets_grouped[current_dataset]
        total_features = x_train_raw.shape[1]

        # Agrego focal a las opciones de pesos
        weight_func_name = trial.suggest_categorical('weight_func', ['none', 'balanced', 'focal'])
        
        if weight_func_name == 'none':
            weight_dict_fs = None
            sample_weights_xgb = None
            
        elif weight_func_name == 'balanced':
            weight_dict_fs = 'balanced'
            sample_weights_xgb = compute_sample_weight('balanced', y=y_train_raw)
            
        else:
            if weight_func_name == 'focal':
                gamma = trial.suggest_float('focal_gamma', 1.0, 5.0)
                weight_dict_fs = focal_class_weight_improved(y_train_raw, gamma=gamma)

            # Ajusto el intervalo de BotBoost con base en resultados
            web_boost = trial.suggest_float('web_boost', 1.0, 2.0) 
            bot_boost = trial.suggest_float('bot_boost', 1.0, 1.5) 
            
            if 12 in weight_dict_fs: weight_dict_fs[12] *= web_boost
            if 1 in weight_dict_fs: weight_dict_fs[1] *= bot_boost
                
            map_func = np.vectorize(lambda x: weight_dict_fs.get(x, 1.0))
            sample_weights_xgb = map_func(y_train_raw)

        fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
        
        if fs_method == 'none':
            k_features = total_features
        else:
            k_features = trial.suggest_int('k_features', 20, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features, class_weight=weight_dict_fs) 
        x_train_fs = selector.fit_transform(x_train_raw, y_train_raw)
        x_val_fs = selector.transform(X_val_imp_grouped)
        
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 150, 400),
            'max_depth': trial.suggest_int('max_depth', 6, 15), 
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            # Cambio a probabilidades para el objetivo de XGBoost
            'objective': 'multi:softprob',
            'num_class': 13,
            'random_state': 42
        }
        
        model = xgb.XGBClassifier(**xgb_params)
        model.fit(x_train_fs, y_train_raw, sample_weight=sample_weights_xgb)
        
        # Extraigo probabilidades en lugar de la clase final
        y_pred_proba = model.predict_proba(x_val_fs)
        
        # Optimización de Umbrales con Pesos de Decisión
        def threshold_loss(weights):
            weighted_probs = y_pred_proba * weights
            y_pred_tuned = np.argmax(weighted_probs, axis=1)
            return -f1_score(y_val_grouped, y_pred_tuned, average='macro')
        
        initial_weights = np.ones(13)
        bounds = [(0.1, 5.0) for _ in range(13)]
        
        res = minimize(threshold_loss, initial_weights, method='Nelder-Mead', bounds=bounds)
        best_threshold_weights = res.x
        
        final_probs = y_pred_proba * best_threshold_weights
        y_pred = np.argmax(final_probs, axis=1)
        
        f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
        report = classification_report(y_val_grouped, y_pred, zero_division=0)
        
        trial.set_user_attr("best_threshold_weights", best_threshold_weights.tolist())
        
        save_confusion_matrix_xgb(log_folder, y_val_grouped, y_pred, trial.number, current_dataset, phase="val")
        
        log_msg = f"""dataset: {current_dataset} | trial: {trial.number}
fs method: {fs_method} | k features: {k_features}
params xgb: {trial.params}
best threshold weights: {np.round(best_threshold_weights, 3).tolist()}

f1 macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)
            
        return f1_mac

    study_name = f"xgb_optimizado_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_xgb, n_trials=75, n_jobs=1) #Genero 75 entrenamientos para ir viendo como va convergiendo
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params,
        'best_weights': study.best_trial.user_attrs["best_threshold_weights"]
    }
    
    best_log_msg = f"""mejor trial para dataset agrupado (xgboost optimizado): {current_dataset}
trial numero: {study.best_trial.number}
f1 macro alcanzado: {study.best_value:.4f}
hiperparametros:
{study.best_params}
best threshold weights:
{np.round(study.best_trial.user_attrs["best_threshold_weights"], 3).tolist()}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)

    print(f"\nResultados finales para {current_dataset}")
    print(f"Mejor trial: {study.best_trial.number}")
    print(f"Mejor F1 Macro: {study.best_value:.4f}\n")

[I 2026-03-15 18:03:08,025] A new study created in memory with name: xgb_optimizado_none



Iniciando estudio enfocado en set agrupado con XGBoost: none


[I 2026-03-15 18:04:14,028] Trial 0 finished with value: 0.9578913375423106 and parameters: {'weight_func': 'balanced', 'fs_method': 'none', 'n_estimators': 246, 'max_depth': 9, 'learning_rate': 0.040365419041749125, 'subsample': 0.9457850194855779, 'colsample_bytree': 0.6408367408611916}. Best is trial 0 with value: 0.9578913375423106.
[I 2026-03-15 18:05:50,476] Trial 1 finished with value: 0.6007249030820966 and parameters: {'weight_func': 'focal', 'focal_gamma': 3.0056977919805434, 'web_boost': 1.4330676808671994, 'bot_boost': 1.4842570377089774, 'fs_method': 'tree', 'k_features': 35, 'n_estimators': 232, 'max_depth': 14, 'learning_rate': 0.10870582935372186, 'subsample': 0.9430457702876628, 'colsample_bytree': 0.6955759083608777}. Best is trial 0 with value: 0.9578913375423106.
[I 2026-03-15 18:07:22,523] Trial 2 finished with value: 0.39491344062145445 and parameters: {'weight_func': 'focal', 'focal_gamma': 3.9354978300070083, 'web_boost': 1.322348163713563, 'bot_boost': 1.399869


Resultados finales para none
Mejor trial: 22
Mejor F1 Macro: 0.9670


Iniciando estudio enfocado en set agrupado con XGBoost: smote


[I 2026-03-15 19:13:18,282] Trial 0 finished with value: 0.9652553415679237 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 279, 'max_depth': 15, 'learning_rate': 0.014552670528330423, 'subsample': 0.6867273785104434, 'colsample_bytree': 0.7740504002849703}. Best is trial 0 with value: 0.9652553415679237.
[I 2026-03-15 19:14:33,546] Trial 1 finished with value: 0.9179480742166691 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 23, 'n_estimators': 368, 'max_depth': 10, 'learning_rate': 0.19973435694692576, 'subsample': 0.933543309812748, 'colsample_bytree': 0.7797415130313956}. Best is trial 0 with value: 0.9652553415679237.
[I 2026-03-15 19:15:46,694] Trial 2 finished with value: 0.9519158277880148 and parameters: {'weight_func': 'balanced', 'fs_method': 'none', 'n_estimators': 335, 'max_depth': 10, 'learning_rate': 0.04951925675992207, 'subsample': 0.9358936140628089, 'colsample_bytree': 0.94232316294938}. Best is trial 0 wit


Resultados finales para smote
Mejor trial: 74
Mejor F1 Macro: 0.9670


Iniciando estudio enfocado en set agrupado con XGBoost: smote_enn


[I 2026-03-15 20:19:04,786] Trial 0 finished with value: 0.9499709345958626 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 25, 'n_estimators': 319, 'max_depth': 8, 'learning_rate': 0.06382541627249445, 'subsample': 0.657783783326358, 'colsample_bytree': 0.6451924823296621}. Best is trial 0 with value: 0.9499709345958626.
[I 2026-03-15 20:20:19,613] Trial 1 finished with value: 0.9406826683019307 and parameters: {'weight_func': 'balanced', 'fs_method': 'tree', 'k_features': 22, 'n_estimators': 217, 'max_depth': 9, 'learning_rate': 0.056588064394054866, 'subsample': 0.6789101825850983, 'colsample_bytree': 0.643793367154615}. Best is trial 0 with value: 0.9499709345958626.
[I 2026-03-15 20:21:52,070] Trial 2 finished with value: 0.9369223137505851 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 275, 'max_depth': 9, 'learning_rate': 0.06769689723853639, 'subsample': 0.8349309914621222, 'colsample_bytree': 0.8785758492814219}. Bes


Resultados finales para smote_enn
Mejor trial: 17
Mejor F1 Macro: 0.9601


Iniciando estudio enfocado en set agrupado con XGBoost: smote_tomek


[I 2026-03-15 21:40:24,757] Trial 0 finished with value: 0.9646168591377559 and parameters: {'weight_func': 'none', 'fs_method': 'none', 'n_estimators': 307, 'max_depth': 9, 'learning_rate': 0.020593077051939072, 'subsample': 0.6451271052985835, 'colsample_bytree': 0.7743107254637008}. Best is trial 0 with value: 0.9646168591377559.
[I 2026-03-15 21:41:31,336] Trial 1 finished with value: 0.6224610674731035 and parameters: {'weight_func': 'focal', 'focal_gamma': 4.979631609149067, 'web_boost': 1.779262825316832, 'bot_boost': 1.19744994473063, 'fs_method': 'none', 'n_estimators': 281, 'max_depth': 13, 'learning_rate': 0.03890677856195804, 'subsample': 0.6315954895909398, 'colsample_bytree': 0.7160249136911288}. Best is trial 0 with value: 0.9646168591377559.
[I 2026-03-15 21:43:26,317] Trial 2 finished with value: 0.800466876326162 and parameters: {'weight_func': 'focal', 'focal_gamma': 2.2372435611681176, 'web_boost': 1.2965517474213875, 'bot_boost': 1.3378398828448357, 'fs_method': 't


Resultados finales para smote_tomek
Mejor trial: 72
Mejor F1 Macro: 0.9669



In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import gc

print("Iniciando evaluación final en Test Set para los modelos optimizados de XGBoost...\n")

log_folder = "Logs_XGBoost_Baseline_2"
os.makedirs(log_folder, exist_ok=True)

best_xgb_configs = {
    'none': {
        'trial': 22, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'none',
        'best_weights': np.array([0.437, 1.072, 1.071, 1.068, 1.056, 0.989, 1.065, 1.087, 1.033, 1.053, 1.075, 1.072, 1.071]),
        'xgb_params': {
            'n_estimators': 320, 'max_depth': 13, 'learning_rate': 0.035218775854037994, 
            'subsample': 0.9947732555645891, 'colsample_bytree': 0.7581856624783883, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softprob', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote_enn': {
        'trial': 17, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'balanced',
        'best_weights': np.array([4.527, 0.1, 1.639, 0.1, 1.389, 0.1, 0.1, 1.092, 1.146, 1.187, 1.275, 0.1, 0.1]),
        'xgb_params': {
            'n_estimators': 196, 'max_depth': 7, 'learning_rate': 0.016536567437232558, 
            'subsample': 0.9477608386290801, 'colsample_bytree': 0.7600044452666339, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softprob', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote_tomek': {
        'trial': 72, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'none',
        'best_weights': np.array([1.375, 0.259, 0.984, 0.99, 1.083, 1.051, 1.122, 1.035, 1.149, 1.075, 0.896, 0.981, 0.838]),
        'xgb_params': {
            'n_estimators': 394, 'max_depth': 6, 'learning_rate': 0.041471374235179936, 
            'subsample': 0.7888406294762074, 'colsample_bytree': 0.9654830299924513, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softprob', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote': {
        'trial': 74, 
        'fs_method': 'tree', 
        'k_features': 54,
        'weight_func': 'none',
        'best_weights': np.array([1.636, 0.372, 1.049, 0.923, 1.137, 1.251, 0.994, 0.96, 0.655, 1.183, 0.93, 0.984, 0.989]),
        'xgb_params': {
            'n_estimators': 150, 'max_depth': 6, 'learning_rate': 0.11609625802660041, 
            'subsample': 0.7941589780550395, 'colsample_bytree': 0.6596493159548819, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softprob', 
            'num_class': 13, 'random_state': 42
        }
    }
}

for current_dataset, config in best_xgb_configs.items():
    print(f"Evaluando modelo XGBoost campeón para: {current_dataset.upper()}")
    
    x_train_best, y_train_best = train_datasets_grouped[current_dataset]
    
    weight_dict_fs = None
    sample_weights_xgb = None
    
    if config['weight_func'] == 'balanced':
        weight_dict_fs = 'balanced'
        sample_weights_xgb = compute_sample_weight('balanced', y=y_train_best)
        print("Aplicando pesos 'balanced'...")

    print(f"Aplicando Feature Selection: {config['fs_method']} (k={config['k_features']})...")
    final_selector = FeatureSelector(method=config['fs_method'], k=config['k_features'], class_weight=weight_dict_fs)
    x_train_final = final_selector.fit_transform(x_train_best, y_train_best)
    x_test_final = final_selector.transform(X_test_imp_grouped) 
    
    x_train_final = np.ascontiguousarray(x_train_final, dtype=np.float32)
    y_train_best = np.ascontiguousarray(y_train_best, dtype=np.int32)
    x_test_final = np.ascontiguousarray(x_test_final, dtype=np.float32)
    if sample_weights_xgb is not None:
        sample_weights_xgb = np.ascontiguousarray(sample_weights_xgb, dtype=np.float32)

    print("Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..")
    final_xgb = xgb.XGBClassifier(**config['xgb_params'])
    
    if sample_weights_xgb is not None:
        final_xgb.fit(x_train_final, y_train_best, sample_weight=sample_weights_xgb)
    else:
        final_xgb.fit(x_train_final, y_train_best)
    
    print("Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...")
    y_pred_proba_test = final_xgb.predict_proba(x_test_final)
    
    final_probs_test = y_pred_proba_test * config['best_weights']
    y_pred_test = np.argmax(final_probs_test, axis=1)
    
    test_f1_mac = f1_score(y_test_grouped, y_pred_test, average='macro')
    test_report = classification_report(y_test_grouped, y_pred_test, zero_division=0)
    
    print(f"Resultados finales - F1 macro: {test_f1_mac:.4f}\n")
    
    save_confusion_matrix_xgb(
        log_folder,
        y_true=y_test_grouped, 
        y_pred=y_pred_test, 
        trial_number=config['trial'], 
        dataset_name=current_dataset, 
        phase="test_final"
    )
    
    log_msg = f"""Evaluación final con Test
Modelo: XGBoost (Tuned Thresholds)
Dataset: {current_dataset} | Feature Selection: {config['fs_method']} (k={config['k_features']})
Weight Function: {config['weight_func']}
Params XGB: {config['xgb_params']}
Best Threshold Weights: {config['best_weights'].tolist()}

F1 macro (Test): {test_f1_mac:.4f}

Métricas por clase (Test):
{test_report}
"""
    log_filename = f"{log_folder}/xgb_final_test_metrics_{current_dataset}.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
        
    del final_xgb
    del x_train_final
    del x_test_final
    del y_pred_proba_test
    del final_probs_test
    gc.collect()

print("Pipeline de pruebas completado con éxito para XGBoost.")

Iniciando evaluación final en Test Set para los CAMPEONES OPTIMIZADOS de XGBoost...

Evaluando modelo XGBoost campeón para: NONE
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...
Resultados finales - F1 macro: 0.9070

Evaluando modelo XGBoost campeón para: SMOTE_ENN
Aplicando pesos 'balanced'...
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...
Resultados finales - F1 macro: 0.9302

Evaluando modelo XGBoost campeón para: SMOTE_TOMEK
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...
Resultados finales - F1 macro: 0.9329

Evaluando modelo XGBoost campeón para: SMOTE
Aplicando Feature Selection

In [17]:
#Ha habido una notable mejora pero voy a continuar haciendo modificaciones sobre mis modelos para validar en experimentos si puedo conseguir seguir mejorando las metricas para esto:
#Voy a eliminar bot_boost y web_boost manuales para que mediante focal loss se pueda identificar los mejores pesos para la clasificacion de las clases que me dan problemas
#Voy a enfocar mi entrenamiento en el uso exclusivo de focal loss

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

log_folder = "Logs_XGBoost_Baseline_3"
os.makedirs(log_folder, exist_ok=True)

# Solo pruebo los mejores datasets ya que none no dio tan buenos resultados en test
datasets_to_test = ['smote_tomek', 'smote_enn', 'smote']
resultados_globales = {}

for current_dataset in datasets_to_test:
    print(f"\nIniciando estudio con Focal Loss en set: {current_dataset}")
    
    def objective_xgb(trial):
        x_train_raw, y_train_raw = train_datasets_grouped[current_dataset]
        total_features = x_train_raw.shape[1]

        gamma = trial.suggest_float('focal_gamma', 0.5, 3.5)
        weight_dict_fs = focal_class_weight_improved(y_train_raw, gamma=gamma)
        
        map_func = np.vectorize(lambda x: weight_dict_fs.get(x, 1.0))
        sample_weights_xgb = map_func(y_train_raw)

        fs_method = trial.suggest_categorical('fs_method', ['none', 'tree'])
        
        if fs_method == 'none':
            k_features = total_features
        else:
            k_features = trial.suggest_int('k_features', 30, total_features)
        
        selector = FeatureSelector(method=fs_method, k=k_features, class_weight=weight_dict_fs) 
        x_train_fs = selector.fit_transform(x_train_raw, y_train_raw)
        x_val_fs = selector.transform(X_val_imp_grouped)
        
        xgb_params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 450),
            'max_depth': trial.suggest_int('max_depth', 6, 12), 
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'subsample': trial.suggest_float('subsample', 0.7, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
            'tree_method': 'hist', 
            'device': 'cuda',
            'objective': 'multi:softprob',
            'num_class': 13,
            'random_state': 42
        }
        
        x_train_fs = np.ascontiguousarray(x_train_fs, dtype=np.float32)
        y_train_raw = np.ascontiguousarray(y_train_raw, dtype=np.int32)
        x_val_fs = np.ascontiguousarray(x_val_fs, dtype=np.float32)
        sample_weights_xgb = np.ascontiguousarray(sample_weights_xgb, dtype=np.float32)

        model = xgb.XGBClassifier(**xgb_params)
        model.fit(x_train_fs, y_train_raw, sample_weight=sample_weights_xgb)
        
        y_pred_proba = model.predict_proba(x_val_fs)
        
        def threshold_loss(weights):
            weighted_probs = y_pred_proba * weights
            y_pred_tuned = np.argmax(weighted_probs, axis=1)
            return -f1_score(y_val_grouped, y_pred_tuned, average='macro')
        
        initial_weights = np.ones(13)
        bounds = [(0.1, 5.0) for _ in range(13)]
        
        res = minimize(threshold_loss, initial_weights, method='Nelder-Mead', bounds=bounds)
        best_threshold_weights = res.x
        
        final_probs = y_pred_proba * best_threshold_weights
        y_pred = np.argmax(final_probs, axis=1)
        
        f1_mac = f1_score(y_val_grouped, y_pred, average='macro')
        report = classification_report(y_val_grouped, y_pred, zero_division=0)
        
        trial.set_user_attr("best_threshold_weights", best_threshold_weights.tolist())
        
        save_confusion_matrix_xgb(log_folder, y_val_grouped, y_pred, trial.number, current_dataset, phase="val")
        
        log_msg = f"""dataset: {current_dataset} | trial: {trial.number}
fs method: {fs_method} | k features: {k_features}
params xgb: {trial.params}
best threshold weights: {np.round(best_threshold_weights, 3).tolist()}

f1 macro: {f1_mac:.4f}
{report}"""
        
        with open(f"{log_folder}/{current_dataset}_trial_{trial.number}.log", 'w') as f:
            f.write(log_msg)

        del model
        del selector
        del x_train_fs
        del x_val_fs
        del y_pred_proba
        del final_probs
        gc.collect()
            
        return f1_mac

    study_name = f"xgb_focal_{current_dataset}"
    study = optuna.create_study(direction='maximize', study_name=study_name)
    study.optimize(objective_xgb, n_trials=100, n_jobs=1, gc_after_trial=True) 
    
    resultados_globales[current_dataset] = {
        'best_trial': study.best_trial.number,
        'best_f1_macro': study.best_value,
        'best_params': study.best_params,
        'best_weights': study.best_trial.user_attrs["best_threshold_weights"]
    }
    
    best_log_msg = f"""Mejor trial para dataset agrupado: {current_dataset}
Trial numero: {study.best_trial.number}
F1 macro alcanzado: {study.best_value:.4f}
Hiperparametros:
{study.best_params}
best threshold weights:
{np.round(study.best_trial.user_attrs["best_threshold_weights"], 3).tolist()}
"""
    with open(f"{log_folder}/mejor_trial_{current_dataset}.log", 'w') as f:
        f.write(best_log_msg)

    print(f"\nResultados finales para {current_dataset}")
    print(f"Mejor trial: {study.best_trial.number}")
    print(f"Mejor F1 Macro: {study.best_value:.4f}\n")

[I 2026-03-16 00:30:28,620] A new study created in memory with name: xgb_focal_smote_tomek



Iniciando estudio con Focal Loss en set: smote_tomek


[I 2026-03-16 00:32:05,626] Trial 0 finished with value: 0.7982521879356678 and parameters: {'focal_gamma': 3.480008206945233, 'fs_method': 'tree', 'k_features': 67, 'n_estimators': 292, 'max_depth': 8, 'learning_rate': 0.1104488042829889, 'subsample': 0.7961862735824342, 'colsample_bytree': 0.8019355611229535}. Best is trial 0 with value: 0.7982521879356678.
[I 2026-03-16 00:33:43,870] Trial 1 finished with value: 0.9189277466732422 and parameters: {'focal_gamma': 1.0105456196626426, 'fs_method': 'tree', 'k_features': 67, 'n_estimators': 281, 'max_depth': 11, 'learning_rate': 0.045256323000805204, 'subsample': 0.9711296579071816, 'colsample_bytree': 0.8858773202362273}. Best is trial 1 with value: 0.9189277466732422.
[I 2026-03-16 00:35:13,535] Trial 2 finished with value: 0.9057403953120543 and parameters: {'focal_gamma': 1.519319613856776, 'fs_method': 'tree', 'k_features': 51, 'n_estimators': 448, 'max_depth': 6, 'learning_rate': 0.09133253696389859, 'subsample': 0.7138278572151484


Resultados finales para smote_tomek
Mejor trial: 21
Mejor F1 Macro: 0.9334


Iniciando estudio con Focal Loss en set: smote_enn


[I 2026-03-16 01:50:31,163] Trial 0 finished with value: 0.7983166570660847 and parameters: {'focal_gamma': 3.187077225981017, 'fs_method': 'none', 'n_estimators': 210, 'max_depth': 7, 'learning_rate': 0.08952695891857208, 'subsample': 0.7992857417626023, 'colsample_bytree': 0.7723554924109357}. Best is trial 0 with value: 0.7983166570660847.
[I 2026-03-16 01:51:21,628] Trial 1 finished with value: 0.7808211755176248 and parameters: {'focal_gamma': 3.053409839267728, 'fs_method': 'tree', 'k_features': 47, 'n_estimators': 230, 'max_depth': 6, 'learning_rate': 0.1203559758599284, 'subsample': 0.7960152001905988, 'colsample_bytree': 0.9381577159388133}. Best is trial 0 with value: 0.7983166570660847.
[I 2026-03-16 01:52:06,467] Trial 2 finished with value: 0.8232475217915773 and parameters: {'focal_gamma': 2.7714853681415748, 'fs_method': 'none', 'n_estimators': 437, 'max_depth': 12, 'learning_rate': 0.04747080516352672, 'subsample': 0.799424196038691, 'colsample_bytree': 0.99021400003432


Resultados finales para smote_enn
Mejor trial: 11
Mejor F1 Macro: 0.9485


Iniciando estudio con Focal Loss en set: smote


[I 2026-03-16 03:56:42,655] Trial 0 finished with value: 0.8893131207347819 and parameters: {'focal_gamma': 1.3168322762092415, 'fs_method': 'tree', 'k_features': 46, 'n_estimators': 318, 'max_depth': 11, 'learning_rate': 0.02010509734742778, 'subsample': 0.9464999521639897, 'colsample_bytree': 0.7625230486972943}. Best is trial 0 with value: 0.8893131207347819.
[I 2026-03-16 03:57:52,255] Trial 1 finished with value: 0.8541726160174288 and parameters: {'focal_gamma': 1.7761648767979614, 'fs_method': 'none', 'n_estimators': 333, 'max_depth': 12, 'learning_rate': 0.019859718031161662, 'subsample': 0.8375181583594526, 'colsample_bytree': 0.930423300832919}. Best is trial 0 with value: 0.8893131207347819.
[I 2026-03-16 03:58:54,342] Trial 2 finished with value: 0.6499684777524142 and parameters: {'focal_gamma': 2.276023235923494, 'fs_method': 'none', 'n_estimators': 213, 'max_depth': 10, 'learning_rate': 0.011661092021228314, 'subsample': 0.9810625914552686, 'colsample_bytree': 0.96129990

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import xgboost as xgb
import numpy as np
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
import gc

print("Iniciando evaluación final en Test Set para los modelos optimizados de XGBoost...\n")

log_folder = "Logs_XGBoost_Baseline_3"
os.makedirs(log_folder, exist_ok=True)

best_xgb_configs = {
    'smote_enn': {
        'trial': 11, 
        'fs_method': 'tree', 
        'k_features': 63, 
        'weight_func': 'focal',
        'focal_gamma': 0.6947158176600201,
        'best_weights': np.array([4.974, 0.1, 3.922, 0.174, 2.3, 0.278, 0.1, 0.129, 0.32, 0.284, 4.041, 0.1, 0.103]),
        'xgb_params': {
            'n_estimators': 311, 'max_depth': 10, 'learning_rate': 0.011108264780432779, 
            'subsample': 0.9915807881024017, 'colsample_bytree': 0.7063229958356859, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softprob', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote_tomek': {
        'trial': 21, 
        'fs_method': 'none', 
        'k_features': 80, 
        'weight_func': 'focal',
        'focal_gamma': 0.5277269777243365,
        'best_weights': np.array([5.0, 0.1, 0.105, 0.128, 3.354, 0.112, 0.173, 0.114, 2.43, 0.1, 4.415, 0.119, 0.1]),
        'xgb_params': {
            'n_estimators': 227, 'max_depth': 7, 'learning_rate': 0.03239897241511407, 
            'subsample': 0.9074890050546547, 'colsample_bytree': 0.9665942713242082, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softprob', 
            'num_class': 13, 'random_state': 42
        }
    },
    'smote': {
        'trial': 86, 
        'fs_method': 'tree', 
        'k_features': 47,
        'weight_func': 'focal',
        'focal_gamma': 0.8692499032509647,
        'best_weights': np.array([4.987, 0.1, 1.066, 0.1, 0.76, 0.1, 0.1, 0.107, 0.781, 0.1, 0.169, 0.1, 0.1]),
        'xgb_params': {
            'n_estimators': 420, 'max_depth': 10, 'learning_rate': 0.03586296994381779, 
            'subsample': 0.9012522863102151, 'colsample_bytree': 0.874868668183697, 
            'tree_method': 'hist', 'device': 'cuda', 'objective': 'multi:softprob', 
            'num_class': 13, 'random_state': 42
        }
    }
}

for current_dataset, config in best_xgb_configs.items():
    print(f"Evaluando modelo XGBoost campeón para: {current_dataset.upper()}")
    
    x_train_best, y_train_best = train_datasets_grouped[current_dataset]
    
    print(f"Aplicando Focal Loss (Gamma: {config['focal_gamma']:.4f})...")
    weight_dict_fs = focal_class_weight_improved(y_train_best, gamma=config['focal_gamma'])
    map_func = np.vectorize(lambda x: weight_dict_fs.get(x, 1.0))
    sample_weights_xgb = map_func(y_train_best)

    print(f"Aplicando Feature Selection: {config['fs_method']} (k={config['k_features']})...")
    final_selector = FeatureSelector(method=config['fs_method'], k=config['k_features'], class_weight=weight_dict_fs)
    x_train_final = final_selector.fit_transform(x_train_best, y_train_best)
    x_test_final = final_selector.transform(X_test_imp_grouped) 
    
    x_train_final = np.ascontiguousarray(x_train_final, dtype=np.float32)
    y_train_best = np.ascontiguousarray(y_train_best, dtype=np.int32)
    x_test_final = np.ascontiguousarray(x_test_final, dtype=np.float32)
    sample_weights_xgb = np.ascontiguousarray(sample_weights_xgb, dtype=np.float32)

    print("Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..")
    final_xgb = xgb.XGBClassifier(**config['xgb_params'])
    
    final_xgb.fit(x_train_final, y_train_best, sample_weight=sample_weights_xgb)
    
    print("Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...")
    y_pred_proba_test = final_xgb.predict_proba(x_test_final)
    
    final_probs_test = y_pred_proba_test * config['best_weights']
    y_pred_test = np.argmax(final_probs_test, axis=1)
    
    test_f1_mac = f1_score(y_test_grouped, y_pred_test, average='macro')
    test_report = classification_report(y_test_grouped, y_pred_test, zero_division=0)
    
    print(f"Resultados finales - F1 macro: {test_f1_mac:.4f}\n")
    
    save_confusion_matrix_xgb(
        log_folder,
        y_true=y_test_grouped, 
        y_pred=y_pred_test, 
        trial_number=config['trial'], 
        dataset_name=current_dataset, 
        phase="test_final"
    )
    
    log_msg = f"""Evaluación final con Test
Modelo: XGBoost (Focal Loss + Tuned Thresholds)
Dataset: {current_dataset} | Feature Selection: {config['fs_method']} (k={config['k_features']})
Weight Function: Focal (Gamma: {config['focal_gamma']:.4f})
Params XGB: {config['xgb_params']}
Best Threshold Weights: {config['best_weights'].tolist()}

F1 macro (Test): {test_f1_mac:.4f}

Métricas por clase (Test):
{test_report}
"""
    log_filename = f"{log_folder}/xgb_final_test_metrics_{current_dataset}.log"
    with open(log_filename, 'w', encoding='utf-8') as f:
        f.write(log_msg)
        
    del final_xgb
    del x_train_final
    del x_test_final
    del y_pred_proba_test
    del final_probs_test
    gc.collect()

print("Pipeline de pruebas completado con éxito para XGBoost.")

Iniciando evaluación final en Test Set para los modelos optimizados de XGBoost...

Evaluando modelo XGBoost campeón para: SMOTE_ENN
Aplicando Focal Loss (Gamma: 0.6947)...
Aplicando Feature Selection: tree (k=63)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...
Resultados finales - F1 macro: 0.9280

Evaluando modelo XGBoost campeón para: SMOTE_TOMEK
Aplicando Focal Loss (Gamma: 0.5277)...
Aplicando Feature Selection: none (k=80)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...
Resultados finales - F1 macro: 0.9188

Evaluando modelo XGBoost campeón para: SMOTE
Aplicando Focal Loss (Gamma: 0.8692)...
Aplicando Feature Selection: tree (k=47)...
Entrenando XGBoost en la GPU 2 (Virtualizada como GPU 0)..
Evaluando probabilidades en el set de test puro y aplicando Threshold Weights...
Resultados finales - F

In [ ]:
#Aqui al revisar los pesos claramente hubo overfitting hacia el set de validación que al pasar a test genero que bajaran un poco las metricas
#El problema sigue siendo la clase 1- Bot, la clase 1 se sigue confundiendo con la clase 0
#Voy a generar nuevos datasets ajustando estos temas desde el oversampling y con eso voy a ejecutar nuevas pruebas